# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Same careful reading applied twice: first to two findings from FlyRank's own research paper
(*The State of AI-Driven SEO, March 2026*), then to my own Week-5 model. The goal isn't to grade
either one — it's to ask the same methodology questions of my own work that I'd want asked of it.

Skills used: `hunting-leakage-and-validating` + `flyrank/flyrank-data` (per `skills/README.md`).

## 1. Two paper findings + my methodology questions

I picked the two ML-appendix findings, since they're the ones closest to what I built myself in
Weeks 5–6 — same kind of model, same kind of question, easiest to hold to the same standard.

### Finding A — "What Predicts Health?" (ML Appendix, Feature Importance page)

The paper trains a Random Forest to predict FlyRank's own composite `health_score` and reports
feature importance: Average Position leads at 43%, then Impressions at 32%, then Scroll Depth at
15%. The paper's own text already flags the catch: `health_score` is *defined* as a weighted sum
of Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — so three of
the top features are literal components of the label, not independent predictors of it. The paper
calls the result "descriptive rather than causal" for exactly this reason.

**My methodology question:** given that the label is arithmetically constructed from the top
three features, what would the ranking look like with those three components removed and only
the *non*-constituent features (Clicks, Sessions, Content Age, Word Count, Days Visible, AI
Sessions) left in? Right now the chart can't distinguish "the model found a real driver of health"
from "the model rediscovered the health-score formula." The paper's own caution is the right
instinct — I'd only push for the ablation to turn that caution into a number the reader can see,
the same "train once WITH the suspect, once WITHOUT" test I run on my own model in Section 3.

### Finding B — "What Predicts Growth?" (ML Appendix, Growth & Classification page)

A logistic regression reports 71% holdout accuracy separating growing from declining pages, with
Content Age and Days Since Update as the strongest negative signals and Days Visible / recent
Impressions among the strongest positive ones. This is functionally the same question as my own
Week-5/6 model (`trend_direction` up vs. down), so I can hold it to the exact standard I hold
myself to.

**My methodology questions, both grounded in the paper's own definitions page:**
1. **Window overlap.** The paper defines trend direction from the 30-day-vs-previous-30-day
   impression change. If "Impressions" as a growth-model feature is the same 90-day total that
   *contains* both of those 30-day sub-windows, the feature partially overlaps the label's own
   definition window — the leakage-taxonomy pattern this week's skill calls "future/overlapping
   windows." I ran exactly this test on my own equivalent features in Section 3 below, and it's
   the single most useful thing I'd ask the paper's authors to check on theirs.
2. **Base rate + split design.** 71% accuracy is reported with no base rate next to it, and no
   detail on whether the holdout was grouped by brand (57 brands contribute pages, and pages from
   the same brand plausibly share hidden structure — the same client-grouping concern the FlyRank
   starter data dictionary raises for this internship's own dataset). Section 2 shows, on my own
   model, how much a plain random holdout can overstate a grouped one — I'd want to know which
   kind of holdout produced this 71%.

Both questions are asked in the spirit the card asks for: not "this number is wrong," but "here's
the check that would make the number sturdier" — the same next-level-of-rigor test I apply to
myself starting now.

In [1]:
# no computation needed for Section 1 — it's a reading exercise.
# Confirming the source referenced above, for the record:
paper_path = "../../docs/flyrank-seo-research-march-2026.pdf"
print("Paper referenced:", paper_path)
print("Finding A: ML Appendix - Feature Importance (page 27) - 'What Predicts Health?'")
print("Finding B: ML Appendix - Growth & Classification (page 29) - 'What Predicts Growth?'")


Paper referenced: ../../docs/flyrank-seo-research-march-2026.pdf
Finding A: ML Appendix - Feature Importance (page 27) - 'What Predicts Health?'
Finding B: ML Appendix - Growth & Classification (page 29) - 'What Predicts Growth?'


## 2. My model under an honest split (before/after)

Re-running my Week-5 logistic regression twice on identical features and identical data — once
under a **naive random row split** (the "before"), once under the **client-grouped split** I
actually used in Week 5 (the "after") — to see exactly how much of my Week-5 number was real
signal versus client memorization.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
pd.set_option("display.width", 200)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col])

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df["is_declining_label"]
groups = df["client_id"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

def fit_eval(X_train, X_test, y_train, y_test, label):
    model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    return {
        "split": label,
        "test_rows": len(X_test),
        "base_rate": round(y_test.mean(), 3),
        "auc": round(roc_auc_score(y_test, p), 3),
        "precision_at_50": round(precision_at_k(y_test.values, p, 50), 3),
    }

# --- BEFORE: naive random row split (client_id ignored) ---
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive, g_tr_naive, g_te_naive = train_test_split(
    X, y, groups, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
overlap_naive = len(set(g_tr_naive) & set(g_te_naive))
before = fit_eval(X_tr_naive, X_te_naive, y_tr_naive, y_te_naive, "BEFORE — naive random split")
before["client_overlap"] = overlap_naive

# --- AFTER: client-grouped split (what Week 5 actually used) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
overlap_grouped = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
after = fit_eval(X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx], "AFTER — client-grouped split")
after["client_overlap"] = overlap_grouped

before_after = pd.DataFrame([before, after])
before_after


,split,test_rows,base_rate,auc,precision_at_50,client_overlap
0,BEFORE — naive random split,6000,0.542,0.711,0.92,31
1,AFTER — client-grouped split,6163,0.511,0.616,0.72,0


**Reading the gap honestly:** the naive random split shares 31 of 32 clients between train
and test (client_overlap=31) and reports AUC 0.711 and precision@50 of **0.92**. The client-grouped
split — zero client overlap — drops to AUC 0.616 and precision@50 of **0.72**. That 0.20-point
precision@50 gap is not noise; it's the model partly memorizing per-client patterns (a client's
typical traffic level, template, or niche) rather than learning something that transfers to a
client it has never seen. The 0.72 number, not 0.92, is the one I'd actually report and the one
that matches what I used in Week 5 — this section just makes the "why not the higher number"
explicit and measured, instead of asserted.

## 3. Leakage audit

The same hunt from the leakage skill, run against this notebook's actual feature set — checking
each item on the attack checklist rather than assuming it passes.

In [3]:
print("--- Checklist item: no label-derived columns in the feature list ---")
leakage_source_cols = {"trend_direction", "trend_pct"}
feature_cols = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES)
assert leakage_source_cols.isdisjoint(feature_cols), "trend_direction/trend_pct leaked into features!"
print("PASS — trend_direction / trend_pct are not in the feature list.\n")

print("--- Checklist item: no product/optimization flags as features ---")
product_flag_style_cols = {"reason_code", "action", "baseline_score", "health_score", "optimization_flag"}
print("PASS —", product_flag_style_cols & feature_cols or "none of these exist in this dataset's feature set.", "\n")

print("--- Checklist item: window-overlap test (feature vs. label window) ---")
print("Suspects: log_impressions_90d, log_clicks_90d, log_sessions_90d, days_with_impressions, days_with_sessions")
print("Why suspect: the label (trend_direction) is defined from impressions_last_30d vs impressions_prev_30d.")
print("The 90-day totals literally SUM both of those sub-windows, so they partially overlap the label's own window.\n")

suspects = {"log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "days_with_impressions", "days_with_sessions"}
numeric_without_suspects = [c for c in NUMERIC_FEATURES if c not in suspects]

def run_variant(numeric_cols, tag):
    Xv = df[numeric_cols + CATEGORICAL_FEATURES].copy()
    pre_v = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
    ])
    gss_v = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    tr_i, te_i = next(gss_v.split(Xv, y, groups=groups))
    m = Pipeline([("pre", pre_v), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
    m.fit(Xv.iloc[tr_i], y.iloc[tr_i])
    p = m.predict_proba(Xv.iloc[te_i])[:, 1]
    auc = roc_auc_score(y.iloc[te_i], p)
    p50 = precision_at_k(y.iloc[te_i].values, p, 50)
    print(f"{tag:22s} AUC={auc:.3f}  precision@50={p50:.3f}  (n_numeric_features={len(numeric_cols)})")
    return auc

auc_with = run_variant(NUMERIC_FEATURES, "WITH suspects:")
auc_without = run_variant(numeric_without_suspects, "WITHOUT suspects:")
print(f"\nAUC drop when suspects removed: {auc_with - auc_without:.3f}")


--- Checklist item: no label-derived columns in the feature list ---
PASS — trend_direction / trend_pct are not in the feature list.

--- Checklist item: no product/optimization flags as features ---
PASS — none of these exist in this dataset's feature set. 

--- Checklist item: window-overlap test (feature vs. label window) ---
Suspects: log_impressions_90d, log_clicks_90d, log_sessions_90d, days_with_impressions, days_with_sessions
Why suspect: the label (trend_direction) is defined from impressions_last_30d vs impressions_prev_30d.
The 90-day totals literally SUM both of those sub-windows, so they partially overlap the label's own window.



WITH suspects:         AUC=0.616  precision@50=0.720  (n_numeric_features=18)


WITHOUT suspects:      AUC=0.549  precision@50=0.520  (n_numeric_features=13)

AUC drop when suspects removed: 0.067


**Verdict: partial, disclosed overlap — not catastrophic leakage.** Removing the
window-overlapping features drops AUC from 0.616 to 0.549 (barely above the coin-flip line, given
a 0.51 base rate) — a real contribution, not the near-1.0-to-0.7 collapse the skill describes as
"the confession" of a fully label-derived feature. That pattern (a moderate, not catastrophic,
drop) is consistent with legitimate signal — pages with more current traffic plausibly *are* more
likely to be past their peak — rather than the feature secretly encoding the answer. Still: because
`impressions_90d` etc. structurally contain the label's own 30-day sub-windows, I'm treating the
0.616 / 0.72 numbers from Section 2 as a soft upper bound, not a clean out-of-window estimate, and
saying so explicitly rather than presenting them as fully leakage-free.

**Checklist recap:**
- [x] No label-derived columns (`trend_direction`, `trend_pct`) in the feature list
- [x] No product/optimization flags as features (this dataset doesn't ship any)
- [x] Window-overlap tested empirically, not just eyeballed — moderate effect, disclosed above
- [x] Split grouped by client (`client_id`), zero overlap confirmed in Section 2
- [x] Base rate printed next to every metric in Sections 2–3
- [x] Top feature importance already sanity-checked in Week 5 (log_impressions/log_clicks led,
      and now Section 3 explains *why* that's plausible rather than alarming)
- [ ] Time-aware split — not possible on this dataset: it's a single 90-day snapshot per row with
      no per-row calendar date, only tiers. Noted as a limitation, not silently skipped.

## 4. Claim rewrite

**My boldest sentence from Week 5** (`w05_model.ipynb`, Section 3): *"logistic regression beats
the Week-4 rule baseline at every K."*

That sentence is true as far as it goes, but after this notebook it goes further than the evidence
carries in two ways: it doesn't say which split produced the number, and it doesn't flag that part
of the model's edge rides on features that partially overlap the label's own definition window.

**Rewrite, in safe language:**

> Under a client-grouped holdout (zero client overlap between train and test), logistic regression
> **ranked** pages more precisely than the Week-4 rule baseline at every K tested — for example,
> precision@50 of 0.72 versus the rule's 0.64, against a 0.51 base rate. This is **decision-support**
> for prioritizing which pages a human reviews first, not a causal or guaranteed prediction that
> any specific page will decline. Some of that lift is **directional but not fully isolated**: a
> few of the model's numeric features (90-day impressions/clicks/sessions totals) structurally
> overlap the 30-day windows the label itself is defined from, and removing them drops the AUC
> from 0.616 to 0.549 — so the honest reading is "the model finds a real but partial signal,
> some of which may reflect the shared definition window rather than an independent driver."

This keeps every number, drops nothing inconvenient, and swaps "beats" for "ranked more precisely
under this specific holdout" — the claim ladder's difference between an assertion and a measured,
disclosed comparison.

In [4]:
# nothing to compute here — Section 4 is the writing exercise the card asks for.
print("Claim rewritten above using observed / measured / directional / decision-support language.")


Claim rewritten above using observed / measured / directional / decision-support language.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id` / `client_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.